# SpaceX Falcon 9 First Stage Landing Prediction
## Lab 1: Data Collection with the SpaceX API

SpaceX advertises Falcon 9 launches at about 62 million dollars, far below other providers, largely because the first stage can be reused. If we can predict whether the first stage will land, we can estimate the cost of a launch.

**Objectives**
- Request launch data from the SpaceX REST API
- Clean the requested data and keep only Falcon 9 launches
- Deal with missing values and export the result

In [1]:
import requests
import pandas as pd
import numpy as np
import datetime

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

### Helper functions
The launch records only store IDs for the rocket, launchpad, payload and core. These functions call the API again to turn each ID into useful information.

In [2]:
import time

_cache = {}

def get_json(url, retries=4):
    """GET a URL and return its JSON. Reuses earlier answers and retries when the server does not return JSON."""
    if url in _cache:
        return _cache[url]
    last_problem = None
    for attempt in range(retries):
        try:
            r = requests.get(url, timeout=60)
            if r.status_code == 200:
                _cache[url] = r.json()
                return _cache[url]
            last_problem = "status code {}: {}".format(r.status_code, str(r.text)[:150])
        except Exception as e:
            last_problem = repr(e)
        time.sleep(2 * (attempt + 1))  # wait a little longer before each retry
    raise RuntimeError("Could not get JSON from {} ({})".format(url, last_problem))

# From the rocket ID: booster name
def getBoosterVersion(data):
    for x in data['rocket']:
        if x:
            response = get_json("https://api.spacexdata.com/v4/rockets/" + str(x))
            BoosterVersion.append(response['name'])

# From the launchpad ID: launch site name, longitude, latitude
def getLaunchSite(data):
    for x in data['launchpad']:
        if x:
            response = get_json("https://api.spacexdata.com/v4/launchpads/" + str(x))
            Longitude.append(response['longitude'])
            Latitude.append(response['latitude'])
            LaunchSite.append(response['name'])

# From the payload ID: payload mass and target orbit
def getPayloadData(data):
    for load in data['payloads']:
        if load:
            response = get_json("https://api.spacexdata.com/v4/payloads/" + load)
            PayloadMass.append(response['mass_kg'])
            Orbit.append(response['orbit'])

# From the core data: landing outcome, landing type, flights, grid fins, reuse, legs, landing pad, block, serial
def getCoreData(data):
    for core in data['cores']:
        if core['core'] != None:
            response = get_json("https://api.spacexdata.com/v4/cores/" + core['core'])
            Block.append(response['block'])
            ReusedCount.append(response['reuse_count'])
            Serial.append(response['serial'])
        else:
            Block.append(None)
            ReusedCount.append(None)
            Serial.append(None)
        Outcome.append(str(core['landing_success']) + ' ' + str(core['landing_type']))
        Flights.append(core['flight'])
        GridFins.append(core['gridfins'])
        Reused.append(core['reused'])
        Legs.append(core['legs'])
        LandingPad.append(core['landpad'])

### Task 1: Request and parse the SpaceX launch data
First check that the live API answers.

In [3]:
spacex_url = "https://api.spacexdata.com/v4/launches/past"
try:
    response = requests.get(spacex_url, timeout=60)
    print("Live API status code:", response.status_code)
except Exception as e:
    print("Live API could not be reached:", repr(e))

Live API status code: 522


To keep the results consistent, the course provides a static copy of the same API response. We use it for the rest of the lab.

In [4]:
static_json_url = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/API_call_spacex_api.json'

def fetch_launches():
    """Return the launch records as JSON: the static copy first, the live API if the copy is unavailable."""
    for name, url in [('Static copy', static_json_url), ('Live API', spacex_url)]:
        try:
            r = requests.get(url, timeout=60)
            print(name, "status code:", r.status_code)
            if r.status_code == 200:
                launches = r.json()
                print("Using:", name, "-", len(launches), "launch records")
                return launches
            print(name, "did not return data:", str(r.text)[:150])
        except Exception as e:
            print(name, "failed:", repr(e))
    raise RuntimeError("Neither the static copy nor the live SpaceX API returned launch data.")

launches = fetch_launches()

Static copy status code: 200
Using: Static copy - 107 launch records


In [5]:
# Turn the JSON into a flat dataframe
data = pd.json_normalize(launches)
data.head()

,static_fire_date_utc,static_fire_date_unix,tbd,net,window,rocket,success,details,crew,ships,capsules,payloads,launchpad,auto_update,failures,flight_number,name,date_utc,date_unix,date_local,date_precision,upcoming,cores,id,fairings.reused,fairings.recovery_attempt,fairings.recovered,fairings.ships,links.patch.small,links.patch.large,links.reddit.campaign,links.reddit.launch,links.reddit.media,links.reddit.recovery,links.flickr.small,links.flickr.original,links.presskit,links.webcast,links.youtube_id,links.article,links.wikipedia,fairings
0,2006-03-17T00:00:00.000Z,1.142554e+09,False,False,0.0,5e9d0d95eda69955f709d1eb,False,Engine failure at 33 seconds and loss of vehicle,[],[],[],[5eb0e4b5b6c3bb0006eeb1e1],5e9e4502f5090995de566f86,True,"[{'time': 33, 'altitude': None, 'reason': 'merlin engine failure'}]",1,FalconSat,2006-03-24T22:30:00.000Z,1143239400,2006-03-25T10:30:00+12:00,hour,False,"[{'core': '5e9e289df35918033d3b2623', 'flight': 1, 'gridfins': False, 'legs': False, 'reused': False, 'landing_attempt': False, 'landing_success': None, 'landing_type': None, 'landpad': None}]",5eb87cd9ffd86e000604b32a,False,False,False,[],https://images2.imgbox.com/3c/0e/T8iJcSN3_o.png,https://images2.imgbox.com/40/e3/GypSkayF_o.png,None,None,None,None,[],[],None,https://www.youtube.com/watch?v=0a_00nJ_Y88,0a_00nJ_Y88,https://www.space.com/2196-spacex-inaugural-falcon-1-rocket-lost-launch.html,https://en.wikipedia.org/wiki/DemoSat,NaN
1,None,NaN,False,False,0.0,5e9d0d95eda69955f709d1eb,False,"Successful first stage burn and transition to second stage, maximum altitude 289 km, Premature engine shutdown at T+7 min 30 s, Failed to reach orbit, Failed to recover first stage",[],[],[],[5eb0e4b6b6c3bb0006eeb1e2],5e9e4502f5090995de566f86,True,"[{'time': 301, 'altitude': 289, 'reason': 'harmonic oscillation leading to premature engine shutdown'}]",2,DemoSat,2007-03-21T01:10:00.000Z,1174439400,2007-03-21T13:10:00+12:00,hour,False,"[{'core': '5e9e289ef35918416a3b2624', 'flight': 1, 'gridfins': False, 'legs': False, 'reused': False, 'landing_attempt': False, 'landing_success': None, 'landing_type': None, 'landpad': None}]",5eb87cdaffd86e000604b32b,False,False,False,[],https://images2.imgbox.com/4f/e3/I0lkuJ2e_o.png,https://images2.imgbox.com/be/e7/iNqsqVYM_o.png,None,None,None,None,[],[],None,https://www.youtube.com/watch?v=Lk4zQ2wP-Nc,Lk4zQ2wP-Nc,https://www.space.com/3590-spacex-falcon-1-rocket-fails-reach-orbit.html,https://en.wikipedia.org/wiki/DemoSat,NaN
2,None,NaN,False,False,0.0,5e9d0d95eda69955f709d1eb,False,Residual stage 1 thrust led to collision between stage 1 and stage 2,[],[],[],"[5eb0e4b6b6c3bb0006eeb1e3, 5eb0e4b6b6c3bb0006eeb1e4]",5e9e4502f5090995de566f86,True,"[{'time': 140, 'altitude': 35, 'reason': 'residual stage-1 thrust led to collision between stage 1 and stage 2'}]",3,Trailblazer,2008-08-03T03:34:00.000Z,1217734440,2008-08-03T15:34:00+12:00,hour,False,"[{'core': '5e9e289ef3591814873b2625', 'flight': 1, 'gridfins': False, 'legs': False, 'reused': False, 'landing_attempt': False, 'landing_success': None, 'landing_type': None, 'landpad': None}]",5eb87cdbffd86e000604b32c,False,False,False,[],https://images2.imgbox.com/3d/86/cnu0pan8_o.png,https://images2.imgbox.com/4b/bd/d8UxLh4q_o.png,None,None,None,None,[],[],None,https://www.youtube.com/watch?v=v0w9p3U8860,v0w9p3U8860,http://www.spacex.com/news/2013/02/11/falcon-1-flight-3-mission-summary,https://en.wikipedia.org/wiki/Trailblazer_(satellite),NaN
3,2008-09-20T00:00:00.000Z,1.221869e+09,False,False,0.0,5e9d0d95eda69955f709d1eb,True,"Ratsat was carried to orbit on the first successful orbital launch of any privately funded and developed, liquid-propelled carrier rocket, the SpaceX Falcon 1",[],[],[],[5eb0e4b7b6c3bb0006eeb1e5],5e9e4502f5090995de566f86,True,[],4,RatSat,2008-09-28T23:15:00.000Z,1222643700,2008-09-28T11:15:00+12:00,hour,False,"[{'core': '5e9e289ef3591855dc3b2626', 'flight': 1, 'gridfins': False, 'legs': False, 'reused': False, 'landing_attempt': False, 'landing_succes

In [6]:
# Keep only the features we need, plus flight number and date
data = data[['rocket', 'payloads', 'launchpad', 'cores', 'flight_number', 'date_utc']]

# Remove rows with multiple cores (Falcon Heavy) and rows with multiple payloads
data = data[data['cores'].map(len) == 1]
data = data[data['payloads'].map(len) == 1]

# Cores and payloads are lists of size 1, so extract the single value
data['cores'] = data['cores'].map(lambda x: x[0])
data['payloads'] = data['payloads'].map(lambda x: x[0])

# Convert date_utc to a date and restrict the launches to those up to 13 November 2020
data['date'] = pd.to_datetime(data['date_utc']).dt.date
data = data[data['date'] <= datetime.date(2020, 11, 13)]
print(data.shape)
data.head()

(94, 7)


,rocket,payloads,launchpad,cores,flight_number,date_utc,date
0,5e9d0d95eda69955f709d1eb,5eb0e4b5b6c3bb0006eeb1e1,5e9e4502f5090995de566f86,"{'core': '5e9e289df35918033d3b2623', 'flight': 1, 'gridfins': False, 'legs': False, 'reused': False, 'landing_attempt': False, 'landing_success': None, 'landing_type': None, 'landpad': None}",1,2006-03-24T22:30:00.000Z,2006-03-24
1,5e9d0d95eda69955f709d1eb,5eb0e4b6b6c3bb0006eeb1e2,5e9e4502f5090995de566f86,"{'core': '5e9e289ef35918416a3b2624', 'flight': 1, 'gridfins': False, 'legs': False, 'reused': False, 'landing_attempt': False, 'landing_success': None, 'landing_type': None, 'landpad': None}",2,2007-03-21T01:10:00.000Z,2007-03-21
3,5e9d0d95eda69955f709d1eb,5eb0e4b7b6c3bb0006eeb1e5,5e9e4502f5090995de566f86,"{'core': '5e9e289ef3591855dc3b2626', 'flight': 1, 'gridfins': False, 'legs': False, 'reused': False, 'landing_attempt': False, 'landing_success': None, 'landing_type': None, 'landpad': None}",4,2008-09-28T23:15:00.000Z,2008-09-28
4,5e9d0d95eda69955f709d1eb,5eb0e4b7b6c3bb0006eeb1e6,5e9e4502f5090995de566f86,"{'core': '5e9e289ef359184f103b2627', 'flight': 1, 'gridfins': False, 'legs': False, 'reused': False, 'landing_attempt': False, 'landing_success': None, 'landing_type': None, 'landpad': None}",5,2009-07-13T03:35:00.000Z,2009-07-13
5,5e9d0d95eda69973a809d1ec,5eb0e4b7b6c3bb0006eeb1e7,5e9e4501f509094ba4566f84,"{'core': '5e9e289ef359185f2b3b2628', 'flight': 1, 'gridfins': False, 'legs': False, 'reused': False, 'landing_attempt': False, 'landing_success': None, 'landing_type': None, 'landpad': None}",6,2010-06-04T18:45:00.000Z,2010-06-04


In [7]:
# Global lists filled by the helper functions
BoosterVersion = []
PayloadMass = []
Orbit = []
LaunchSite = []
Outcome = []
Flights = []
GridFins = []
Reused = []
Legs = []
LandingPad = []
Block = []
ReusedCount = []
Serial = []
Longitude = []
Latitude = []

In [9]:
# The helper functions look up every rocket, launchpad, payload and core ID on api.spacexdata.com.
# If that server is down, the same values are taken from the course's saved copy of the API
# results (dataset_part_1.csv), matched to each launch by its date.

all_lists = [BoosterVersion, PayloadMass, Orbit, LaunchSite, Outcome, Flights, GridFins, Reused,
             Legs, LandingPad, Block, ReusedCount, Serial, Longitude, Latitude]
for lst in all_lists:
    lst.clear()  # start clean in case an earlier run stopped half way

def api_is_up():
    """Quick check: can one rocket ID be looked up on the live API?"""
    try:
        r = requests.get("https://api.spacexdata.com/v4/rockets/" + str(data['rocket'].iloc[0]), timeout=20)
        return r.status_code == 200
    except Exception:
        return False

if api_is_up():
    # These calls query the API once per distinct ID, so this can take a minute
    getBoosterVersion(data)
    getLaunchSite(data)
    getPayloadData(data)
    getCoreData(data)
    print("Source: live SpaceX API")
else:
    print("The SpaceX API is not reachable. Using the course's saved copy of the API results.")
    saved = pd.read_csv("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_1.csv")
    saved['Date'] = pd.to_datetime(saved['Date']).dt.date
    saved = saved.drop_duplicates('Date').set_index('Date')

    # One row per launch in `data`; launches missing from the saved copy (the Falcon 1 flights) become NaN
    matched = saved.reindex(list(data['date']))
    found = matched['BoosterVersion'].notna().to_numpy()
    if found.sum() == 0:
        raise RuntimeError("No launch dates matched the saved copy, so the lookup values cannot be filled in.")

    BoosterVersion.extend(np.where(found, matched['BoosterVersion'], 'Falcon 1').tolist())
    columns = {'PayloadMass': PayloadMass, 'Orbit': Orbit, 'LaunchSite': LaunchSite, 'Outcome': Outcome,
               'Flights': Flights, 'GridFins': GridFins, 'Reused': Reused, 'Legs': Legs,
               'LandingPad': LandingPad, 'Block': Block, 'ReusedCount': ReusedCount, 'Serial': Serial,
               'Longitude': Longitude, 'Latitude': Latitude}
    for column, lst in columns.items():
        lst.extend(matched[column].tolist())
    print("Launches matched by date:", int(found.sum()), "of", len(data))

print("Booster versions collected:", len(BoosterVersion))
BoosterVersion[0:5]

The SpaceX API is not reachable. Using the course's saved copy of the API results.
Launches matched by date: 90 of 94
Booster versions collected: 94


['Falcon 1', 'Falcon 1', 'Falcon 1', 'Falcon 1', 'Falcon 9']

In [10]:
launch_dict = {'FlightNumber': list(data['flight_number']),
               'Date': list(data['date']),
               'BoosterVersion': BoosterVersion,
               'PayloadMass': PayloadMass,
               'Orbit': Orbit,
               'LaunchSite': LaunchSite,
               'Outcome': Outcome,
               'Flights': Flights,
               'GridFins': GridFins,
               'Reused': Reused,
               'Legs': Legs,
               'LandingPad': LandingPad,
               'Block': Block,
               'ReusedCount': ReusedCount,
               'Serial': Serial,
               'Longitude': Longitude,
               'Latitude': Latitude}

df = pd.DataFrame(launch_dict)
print(df.shape)
df.head()

(94, 17)


,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude
0,1,2006-03-24,Falcon 1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,2007-03-21,Falcon 1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,4,2008-09-28,Falcon 1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,5,2009-07-13,Falcon 1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,6,2010-06-04,Falcon 9,6104.959412,LEO,CCAFS SLC 40,None None,1.0,False,False,False,NaN,1.0,0.0,B0003,-80.577366,28.561857


### Task 2: Keep only Falcon 9 launches

In [11]:
print(df['BoosterVersion'].value_counts())

data_falcon9 = df[df['BoosterVersion'] != 'Falcon 1'].copy()

# Reset the flight numbers so they run 1, 2, 3, ...
data_falcon9.loc[:, 'FlightNumber'] = list(range(1, data_falcon9.shape[0] + 1))
print(data_falcon9.shape)
data_falcon9.head()

BoosterVersion
Falcon 9    90
Falcon 1     4
Name: count, dtype: int64
(90, 17)


,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude
4,1,2010-06-04,Falcon 9,6104.959412,LEO,CCAFS SLC 40,None None,1.0,False,False,False,NaN,1.0,0.0,B0003,-80.577366,28.561857
5,2,2012-05-22,Falcon 9,525.000000,LEO,CCAFS SLC 40,None None,1.0,False,False,False,NaN,1.0,0.0,B0005,-80.577366,28.561857
6,3,2013-03-01,Falcon 9,677.000000,ISS,CCAFS SLC 40,None None,1.0,False,False,False,NaN,1.0,0.0,B0007,-80.577366,28.561857
7,4,2013-09-29,Falcon 9,500.000000,PO,VAFB SLC 4E,False Ocean,1.0,False,False,False,NaN,1.0,0.0,B1003,-120.610829,34.632093
8,5,2013-12-03,Falcon 9,3170.000000,GTO,CCAFS SLC 40,None None,1.0,False,False,False,NaN,1.0,0.0,B1004,-80.577366,28.561857


### Task 3: Deal with missing values
`LandingPad` keeps its missing values (they mean no landing pad was used). Missing `PayloadMass` values are replaced with the column mean.

In [12]:
data_falcon9.isnull().sum()

,0
FlightNumber,0
Date,0
BoosterVersion,0
PayloadMass,0
Orbit,0
LaunchSite,0
Outcome,0
Flights,0
GridFins,0
Reused,0


In [13]:
payload_mean = data_falcon9['PayloadMass'].mean()
print("Mean payload mass (kg):", round(payload_mean, 2))

data_falcon9['PayloadMass'] = data_falcon9['PayloadMass'].replace(np.nan, payload_mean)
data_falcon9.isnull().sum()

Mean payload mass (kg): 6104.96


,0
FlightNumber,0
Date,0
BoosterVersion,0
PayloadMass,0
Orbit,0
LaunchSite,0
Outcome,0
Flights,0
GridFins,0
Reused,0


### Export the result

In [14]:
data_falcon9.to_csv('dataset_part_1.csv', index=False)
print("Saved dataset_part_1.csv with", data_falcon9.shape[0], "rows and", data_falcon9.shape[1], "columns")

Saved dataset_part_1.csv with 90 rows and 17 columns
